In [1]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from src.preprocessing import (build_preprocessor, feature_names,
                               prepare_features, ALL_FEATURES, TARGET)

df = pd.read_parquet("../data/processed/cleaned.parquet")
X, y = df[ALL_FEATURES], df[TARGET]
print("features:", len(ALL_FEATURES))        # expect 29

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train.to_parquet("../data/processed/X_train.parquet")
X_test.to_parquet("../data/processed/X_test.parquet")
y_train.to_frame().to_parquet("../data/processed/y_train.parquet")
y_test.to_frame().to_parquet("../data/processed/y_test.parquet")

features: 29


In [2]:
print("train:", X_train.shape, " test:", X_test.shape)
print("positive rate  train: %.4f   test: %.4f   overall: %.4f"
      % (y_train.mean(), y_test.mean(), y.mean()))
assert len(set(X_train.index) & set(X_test.index)) == 0     # no row in both

train: (361971, 29)  test: (90493, 29)
positive rate  train: 0.0936   test: 0.0936   overall: 0.0936


In [3]:
prep = build_preprocessor()
A_train = prep.fit_transform(X_train)      # learns medians, categories etc. from TRAIN only
A_test  = prep.transform(X_test)           # applies them to test, without learning

print("encoded shapes:", A_train.shape, A_test.shape)
print("NaNs left  train:", np.isnan(A_train).sum(), "  test:", np.isnan(A_test).sum())

names = feature_names(prep)
print(len(names), "model input columns")
print([n for n in names if "missingindicator" in n])

encoded shapes: (361971, 87) (90493, 87)
NaNs left  train: 0   test: 0
87 model input columns
['num__missingindicator__BMI5', 'num__missingindicator_PHYSHLTH', 'num__missingindicator_MENTHLTH', 'num__missingindicator__DRNKWK3', 'num__missingindicator_GENHLTH', 'num__missingindicator_INCOME3', 'num__missingindicator_EDUCA', 'bin__missingindicator_CHCKDNY2', 'bin__missingindicator_CHCCOPD3', 'bin__missingindicator_ASTHMA3', 'bin__missingindicator_ADDEPEV3', 'bin__missingindicator_CHCSCNC1', 'bin__missingindicator_CHCOCNC1', 'bin__missingindicator_HAVARTH4', 'bin__missingindicator_DIFFWALK', 'bin__missingindicator_DIFFDRES', 'bin__missingindicator_DECIDE', 'bin__missingindicator_MEDCOST1', 'bin__missingindicator_DRNKANY6', 'bin__missingindicator__TOTINDA', 'bin__missingindicator__RFDRHV9']


In [4]:
tmp = prepare_features(X_train.copy())
tmp[TARGET] = y_train
print(tmp.groupby("chronic_count")[TARGET].agg(rate="mean", n="size").round(3))
print("\nmax after capping:", tmp[["_BMI5", "_DRNKWK3"]].max().to_dict())

                rate       n
chronic_count               
0.0            0.039  145899
1.0            0.087  112189
2.0            0.137   62284
3.0            0.206   27884
4.0            0.274   10436
5.0            0.378    2694
6.0            0.502     526
7.0            0.661      59

max after capping: {'_BMI5': 70.0, '_DRNKWK3': 100.0}
